# Импорт библиотек

In [2]:
import sys
import os

sys.path.append(os.path.abspath('lib'))

from preprocessing_pipeline import create_combined_pipeline
from test_models import run_models_classifications

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Считывание данных

In [4]:
df = pd.read_csv('data/processed.csv')
df.shape

(998, 213)

# Очистка таргета от выбросов

In [6]:
df = df[df['IC50, mM'] < 4000]

# Подготовка данных для эксперемента

In [8]:
combined_pipeline = create_combined_pipeline()
X = df.drop(columns=['IC50, mM', 'CC50, mM', 'SI'])
y1 = df['IC50, mM']
y2 = df['CC50, mM']
y3 = df['SI']
X_transformed = combined_pipeline.fit_transform(X)
df = pd.concat([X_transformed, y1, y2, y3], axis=1)

приоритет на recall, чтобы не упустить хорошее лекарство

In [10]:
X = df.drop(columns=['IC50, mM', 'CC50, mM', 'SI'])
y = df['IC50, mM'].apply(lambda v: 1 if v >= df['IC50, mM'].median() else 0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Train dataset size: {X_train.shape}, {y_train.shape}')
print(f'Test dataset size: {X_test.shape}, {y_test.shape}')

Train dataset size: (796, 100), (796,)
Test dataset size: (200, 100), (200,)


# Эксперемент с моделями

In [12]:
run_models_classifications(X_train, X_test, y_train, y_test)

,Model,WA f1,accuracy,WA precision,WA recall,WA support,roc_auc
3,Random Forest,0.71,0.71,0.71,0.71,200.0,0.78
9,AdaBoost,0.70,0.69,0.70,0.69,200.0,0.76
4,XGBoost,0.69,0.69,0.69,0.69,200.0,0.74
6,LightGBM,0.69,0.69,0.69,0.69,200.0,0.75
7,CatBoost,0.69,0.69,0.69,0.69,200.0,0.78
2,KNeighbors,0.68,0.69,0.70,0.69,200.0,0.73
5,Gradient Boosting,0.68,0.68,0.68,0.68,200.0,0.75
8,HistGradientBoosting,0.68,0.68,0.68,0.68,200.0,0.74
0,Logistic Regression,0.65,0.66,0.67,0.66,200.0,0.76
1,Decision Tree,0.64,0.64,0.64,0.64,200.0,0.65


# Подбор гиперпараметров

In [14]:
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import classification_report, roc_auc_score
from scipy.stats import randint, uniform

param_dist = {
    'iterations': randint(50, 300),
    'depth': randint(3, 10),
    'learning_rate': uniform(0.01, 0.3),
    'l2_leaf_reg': uniform(1, 10),
    'border_count': randint(32, 255),
    'grow_policy': ['SymmetricTree', 'Depthwise', 'Lossguide'],
    'random_strength': uniform(0, 1),
    'leaf_estimation_iterations': randint(1, 10),
    'bootstrap_type': ['Bayesian', 'Bernoulli', 'MVS'],
    'feature_border_type': ['GreedyLogSum', 'Median', 'Uniform'],
}
random_search = RandomizedSearchCV(
    estimator=CatBoostClassifier(silent=True, random_state=42),
    param_distributions=param_dist,
    n_iter=200,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

best_params = random_search.best_params_
print("Лучшие параметры:", best_params)

/Users/v.papadyk/anaconda3/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Лучшие параметры: {'bootstrap_type': 'MVS', 'border_count': 173, 'depth': 9, 'feature_border_type': 'Uniform', 'grow_policy': 'SymmetricTree', 'iterations': 256, 'l2_leaf_reg': 5.275410183585496, 'leaf_estimation_iterations': 3, 'learning_rate': 0.019428755706020276, 'random_strength': 0.6364104112637804}


In [15]:
model = CatBoostClassifier(
    bootstrap_type='Bayesian',
    border_count=34,
    depth=6,
    feature_border_type='Uniform',
    grow_policy='Lossguide',
    iterations=139,
    l2_leaf_reg=6.12,
    leaf_estimation_iterations=3,
    learning_rate=0.11,
    random_strength=0.01,
    random_state=42,
    verbose=0,
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_score = model.predict_proba(X_test)[:, 1]

report = classification_report(y_test, y_pred, output_dict=True)
report_df = pd.DataFrame(report).transpose()
roc_auc = roc_auc_score(y_test, y_score)

print("Accuracy:", round(report['accuracy'], 2))
print("ROC AUC Score:", round(roc_auc, 2))
report_df

Accuracy: 0.68
ROC AUC Score: 0.77


,precision,recall,f1-score,support
0,0.710280,0.697248,0.703704,109.00
1,0.645161,0.659341,0.652174,91.00
accuracy,0.680000,0.680000,0.680000,0.68
macro avg,0.677721,0.678294,0.677939,200.00
weighted avg,0.680651,0.680000,0.680258,200.00
